# Module 7: Observability & Evaluations -- Monitoring in Production

![Overview](../shared/img/07.drawio.png)

Throughout this workshop, you have been enabling **Tracing** on each AgentCore resource as it was created. The Runtime has been exporting OpenTelemetry traces since Module 2, and Memory and Gateway resources have had tracing enabled since Modules 4 and 5.

In this module, you explore that accumulated data, deploy the final production-hardened version of Aria (V5), then set up **custom evaluators** that continuously score the agent's performance.

## What you will learn

- **Exploring traces**: Navigate the CloudWatch GenAI Observability dashboard to view agent traces, memory operations, and gateway requests
- **Understanding the trace pipeline**: How ADOT instrumentation and X-Ray Transaction Search work together
- **Custom evaluators**: LLM-as-judge evaluators that score response quality and tool usage
- **Online evaluations**: Continuous monitoring with configurable sampling rates

## Catch-up

Ensure all prerequisites from earlier modules are in place.

In [ ]:
import sys; sys.path.insert(0, '..')
from shared.ensure_ready import ensure_ready

config = ensure_ready("07")

## How observability has been working behind the scenes

Every deploy and resource creation in this workshop has included observability configuration. Here's what was set up:

### Agent tracing (Modules 2-6)

Every time you deployed Aria, the deployment helper:
1. Included `aws-opentelemetry-distro` in the agent package
2. Wrapped the entrypoint with `opentelemetry-instrument` to auto-instrument HTTP calls, LLM invocations, and tool calls
3. Set `tracingConfiguration={"enabled": True}` on the Runtime

This means every agent invocation you've run -- from simple conversations in Module 2 to policy-enforced Gateway calls in Module 6 -- has been producing trace data.

### Resource-level tracing (Modules 4-5)

In Modules 4 and 5, you enabled **Tracing** on the Memory and Gateway resources through the AgentCore console. This populates the **Memory** and **Gateways** tabs in the AgentCore Observability dashboard with detailed spans for each operation.

### Service-vended metrics (automatic)

AgentCore automatically emits CloudWatch metrics for every resource -- invocations, latency, errors, throttles, CPU/memory usage. These appear with **zero configuration** in the `Bedrock-AgentCore` namespace.

### Trace flow

![Trace Flow](../shared/img/trace-flow.drawio.png)

> **Documentation:** [AgentCore Observability](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability.html)

To see the full agent code, open [agent/main.py](agent/main.py) in a new tab.

In [ ]:
import sys; sys.path.insert(0, '..')
from shared import utils, deploy_agent

# Gather environment variables from all prior modules
env_vars = {}

memory_config = utils.load_config("memory")
if memory_config:
    env_vars["MEMORY_ID"] = memory_config["memory_id"]

gateway_config = utils.load_config("gateway")
if gateway_config:
    env_vars["GATEWAY_ENDPOINT"] = gateway_config.get("gateway_url", "")

print(f"Environment variables: {list(env_vars.keys())}")
print()

# Deploy V5 (tracing is enabled by default on all deploys)
runtime_config = deploy_agent.deploy(
    agent_dir="agent",
    env_vars=env_vars,
)

## Explore existing traces

You already have trace data in CloudWatch from every agent invocation across Modules 2-6. Let's also generate a few more targeted prompts that exercise different capabilities, then explore everything in the dashboard.

In [ ]:
import sys; sys.path.insert(0, '..')
from shared import test_agent

jwt_token = test_agent.get_test_token()

In [ ]:
# Trace 1: Simple calculation (exercises code interpreter)
result = test_agent.invoke("What is the square root of 144?", jwt_token=jwt_token)

In [ ]:
# Trace 2: Task management (exercises Gateway + Cedar policy)
result = test_agent.invoke(
    "Create a task: Review observability traces in CloudWatch",
    jwt_token=jwt_token,
)

In [ ]:
# Trace 3: Memory storage (exercises memory service)
result = test_agent.invoke(
    "Remember that I prefer dark mode for all my applications",
    jwt_token=jwt_token,
)

In [ ]:
# Trace 4: List tasks (exercises Gateway read operation)
result = test_agent.invoke("Show me all my current tasks", jwt_token=jwt_token)

## Viewing traces in CloudWatch

Open the **CloudWatch Console** and navigate to **Application Signals** > **GenAI Observability** > **Bedrock AgentCore**. You'll see tabs for each resource type:

### Agents tab
Shows all agent invocations with trace details. Click any trace to see the full waterfall:
- **Root span**: The full invocation (request to response)
- **LLM spans**: Each call to the foundation model (with token counts and latency)
- **Tool spans**: Each tool invocation (code interpreter, browser, Gateway tools)
- **Memory spans**: Retrieval and storage operations
- **Gateway spans**: MCP requests to the Gateway (including policy evaluation results)

### Memory tab
Shows Memory resource metrics and (with tracing enabled) detailed spans for:
- `CreateEvent`, `RetrieveMemoryRecords`, `ListMemoryRecords`
- Extraction processing (new memory formation from events)
- Consolidation processing (merging with existing memories)

### Built-in Tools tab
Shows Code Interpreter and Browser metrics:
- Invocation counts, latency, errors
- CPU and memory usage per tool session

### Gateways tab
Shows Gateway metrics and spans for:
- `List Tools`, `Call Tool`, `Search Tools` operations
- Policy evaluation results (allow/deny decisions, determining policies)
- Target execution time (how long the backend API took)

### Other views
- **X-Ray traces** > **Service map**: Visualize the dependency graph between Runtime, LLM, tools, Memory, and Gateway
- **CloudWatch Logs Insights**: Query structured trace data

> **Tip:** You should see traces spanning the entire workshop -- from simple Module 2 conversations through to the policy-enforced Gateway calls in Module 6. The richer your earlier exploration, the more data you have here.

## Custom Evaluators

Observability tells you *what* the agent did. **Evaluations** tell you *how well* it did.

An evaluator is an LLM-as-judge that scores your agent's behavior. You define:
- **Rating scale** (e.g., 1-5)
- **Instructions** -- a rubric the judge follows
- **Evaluation level** -- SESSION (whole conversation) or TRACE (individual turn)

| Concept | Description |
|---|---|
| **Evaluator** | A custom rubric that defines how to score agent behavior |
| **Evaluation level** | **SESSION** evaluates the full conversation. **TRACE** evaluates individual tool calls |
| **Online evaluation** | An evaluator attached to a live agent with a sampling rate |
| **Sampling rate** | Percentage of invocations to evaluate (e.g., 10%). Controls cost and coverage |

> **Docs**: [AgentCore Evaluations](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/evaluations.html)

### Create Evaluator 1: ResponseQuality (SESSION level)

This evaluator examines the full conversation session and scores on a 1-5 scale. We use the **control plane** client (`bedrock-agentcore-control`) to create evaluators.

The evaluator config uses the `llmAsAJudge` structure with:
- **Instructions** containing placeholders that get filled with trace data
- **Rating scale** with numerical values, labels, and definitions
- **Model config** specifying which foundation model acts as the judge

In [ ]:
import boto3
from botocore.exceptions import ClientError
import sys; sys.path.insert(0, '..')
from shared import utils

region = utils.get_region()
control = boto3.client("bedrock-agentcore-control", region_name=region)

# Evaluator 1: Response Quality (SESSION level)
#
# SESSION evaluators require at least one of these placeholders:
#   {available_tools}, {context}, {actual_tool_trajectory},
#   {expected_tool_trajectory}, {assertions}

response_quality_instructions = """You are evaluating the quality of an AI assistant named Aria.

Here is the context of the conversation:
{context}

Here are the tools available to the assistant:
{available_tools}

Here is the actual tool trajectory:
{actual_tool_trajectory}

Score on this scale:
5 - Excellent: Fully addresses request with accurate, complete information.
4 - Good: Mostly complete and accurate with minor gaps.
3 - Adequate: Partially addresses request with noticeable gaps.
2 - Poor: Fails to adequately address request.
1 - Unacceptable: Wrong, hallucinated, or harmful.

Weigh: Accuracy, Completeness, Relevance, Clarity, Groundedness.
Provide brief justification before the numeric rating."""

print("Rubric for ResponseQuality:")
print(response_quality_instructions[:200] + "...")
print()

try:
    resp = control.create_evaluator(
        evaluatorName="ResponseQuality",
        description="Evaluates helpfulness, accuracy, and completeness of responses",
        level="SESSION",
        evaluatorConfig={
            "llmAsAJudge": {
                "instructions": response_quality_instructions,
                "ratingScale": {
                    "numerical": [
                        {"value": 1, "label": "Unacceptable", "definition": "Wrong, hallucinated, or harmful"},
                        {"value": 2, "label": "Poor", "definition": "Fails to adequately address request"},
                        {"value": 3, "label": "Adequate", "definition": "Partially addresses request with noticeable gaps"},
                        {"value": 4, "label": "Good", "definition": "Mostly complete and accurate with minor gaps"},
                        {"value": 5, "label": "Excellent", "definition": "Fully addresses request with accurate, complete information"},
                    ],
                },
                "modelConfig": {
                    "bedrockEvaluatorModelConfig": {
                        "modelId": "us.anthropic.claude-sonnet-4-5-20250929-v1:0",
                    }
                },
            }
        },
    )
    response_quality_id = resp["evaluatorId"]
    print(f"ResponseQuality evaluator created: {response_quality_id}")
    print(f"Status: {resp['status']}")
except ClientError as e:
    if e.response["Error"]["Code"] in ("ConflictException", "ValidationException"):
        print(f"ResponseQuality evaluator already exists or validation issue: {e.response['Error']['Message']}")
        response_quality_id = "existing"
    else:
        raise

### Create Evaluator 2: ToolUsage (TRACE level)

This evaluator examines individual tool calls within a trace and scores whether the agent selected the right tools and used them efficiently.

In [ ]:
# Evaluator 2: Tool Usage (TRACE level)
#
# TRACE evaluators require at least one of these placeholders:
#   {context}, {assistant_turn}, {expected_response}

tool_usage_instructions = """You are evaluating how well an AI assistant named Aria uses its tools.

Here is the context of the interaction:
{context}

Here is the assistant's response:
{assistant_turn}

Score on this scale:
5 - Optimal: Exactly the right tools, well-formed inputs, no unnecessary calls.
4 - Good: Correct tools with minor inefficiencies.
3 - Acceptable: Suboptimal but functional tool usage.
2 - Poor: Wrong tool selected or many unnecessary calls.
1 - Critical failure: Essential tools not used or severe misuse.

Weigh: Tool selection, Input quality, Efficiency, Completeness, Error handling.
Provide brief justification before the numeric rating."""

print("Rubric for ToolUsage:")
print(tool_usage_instructions[:200] + "...")
print()

try:
    resp = control.create_evaluator(
        evaluatorName="ToolUsage",
        description="Evaluates tool selection and usage efficiency",
        level="TRACE",
        evaluatorConfig={
            "llmAsAJudge": {
                "instructions": tool_usage_instructions,
                "ratingScale": {
                    "numerical": [
                        {"value": 1, "label": "Critical failure", "definition": "Essential tools not used or severe misuse"},
                        {"value": 2, "label": "Poor", "definition": "Wrong tool selected or many unnecessary calls"},
                        {"value": 3, "label": "Acceptable", "definition": "Suboptimal but functional tool usage"},
                        {"value": 4, "label": "Good", "definition": "Correct tools with minor inefficiencies"},
                        {"value": 5, "label": "Optimal", "definition": "Exactly the right tools, well-formed inputs, no unnecessary calls"},
                    ],
                },
                "modelConfig": {
                    "bedrockEvaluatorModelConfig": {
                        "modelId": "us.anthropic.claude-sonnet-4-5-20250929-v1:0",
                    }
                },
            }
        },
    )
    tool_usage_id = resp["evaluatorId"]
    print(f"ToolUsage evaluator created: {tool_usage_id}")
    print(f"Status: {resp['status']}")
except ClientError as e:
    if e.response["Error"]["Code"] in ("ConflictException", "ValidationException"):
        print(f"ToolUsage evaluator already exists or validation issue: {e.response['Error']['Message']}")
        tool_usage_id = "existing"
    else:
        raise

## Set up online evaluation

Online evaluations attach evaluators to a live agent with a **sampling rate**. They run asynchronously -- they do not slow down the agent's responses.

Online evaluations require:
- A **data source** pointing to the CloudWatch log group that contains the agent's OTel trace data
- An **execution role** with permissions to read logs and invoke Bedrock models
- **Evaluator references** -- either built-in or custom evaluators

AgentCore provides several built-in evaluators:

| Evaluator | Level | What it measures |
|---|---|---|
| `Builtin.Helpfulness` | TRACE | How helpful the agent's responses are |
| `Builtin.GoalSuccessRate` | SESSION | Whether the agent achieved the user's goal |
| `Builtin.Correctness` | TRACE | Factual accuracy of responses |
| `Builtin.Conciseness` | TRACE | Whether responses are appropriately concise |
| `Builtin.ToolSelectionAccuracy` | TOOL_CALL | Whether the agent chose the right tools |

Below we use the `create_online_evaluation_config` control-plane API to wire up two built-in evaluators to Aria's trace log group. We set the sampling rate to 100% so that evaluation results appear quickly during the workshop. In production, you would typically use a lower rate (e.g., 10%) to control cost.

In [ ]:
# Set up online evaluation using the boto3 control-plane API directly
runtime_config = utils.load_config("runtime")
runtime_id = runtime_config["runtime_id"]
runtime_name = runtime_config["runtime_name"]

# The Runtime's OTel traces are written to this log group automatically
log_group = f"/aws/bedrock-agentcore/runtimes/{runtime_id}-DEFAULT"

# The service name matches the pattern: {runtime_name}.DEFAULT
service_name = f"{runtime_name}.DEFAULT"

# The evaluation execution role is pre-provisioned by the workshop CloudFormation template
cfn_outputs = utils.get_all_cfn_outputs()
eval_role_arn = cfn_outputs["EvaluationRoleArn"]

print(f"Log group:      {log_group}")
print(f"Service name:   {service_name}")
print(f"Execution role: {eval_role_arn}")
print()

try:
    config = control.create_online_evaluation_config(
        onlineEvaluationConfigName="aria_quality_monitor",
        description="Monitor Aria response quality — 100% sampling for workshop",
        rule={
            "samplingConfig": {
                "samplingPercentage": 100.0,
            },
        },
        dataSourceConfig={
            "cloudWatchLogs": {
                "logGroupNames": [log_group],
                "serviceNames": [service_name],
            },
        },
        evaluators=[
            {"evaluatorId": "Builtin.GoalSuccessRate"},
            {"evaluatorId": "Builtin.Helpfulness"},
        ],
        evaluationExecutionRoleArn=eval_role_arn,
        enableOnCreate=True,
    )
    print("Online evaluation created!")
    print(f"  Config ID: {config.get('onlineEvaluationConfigId', 'N/A')}")
    print(f"  Status:    {config.get('status', 'N/A')}")
except ClientError as e:
    if "already exists" in str(e) or e.response["Error"]["Code"] == "ConflictException":
        print(f"Online evaluation config already exists: {e.response['Error']['Message']}")
    else:
        raise

In [ ]:
# List all evaluators (built-in and custom)
evaluators = control.list_evaluators()
print(f"{'Evaluator ID':<45} {'Name':<25} {'Level':<10}")
print("-" * 80)
for e in evaluators.get("evaluators", []):
    eid = e.get("evaluatorId", "?")
    name = e.get("evaluatorName", "?")
    level = e.get("level", "?")
    marker = " <-- custom" if not eid.startswith("Builtin.") else ""
    print(f"{eid:<45} {name:<25} {level:<10}{marker}")

### Save configuration

In [ ]:
# Save evaluations config for Module 8
eval_config = {
    "custom_evaluators": {
        "ResponseQuality": response_quality_id if "response_quality_id" in dir() else "existing",
        "ToolUsage": tool_usage_id if "tool_usage_id" in dir() else "existing",
    },
    "builtin_evaluators": ["Builtin.GoalSuccessRate", "Builtin.Helpfulness"],
}
utils.save_config("evaluations", eval_config)
print("Evaluations configuration saved")

---

## Run on-demand evaluation with custom evaluators

The custom evaluators we created (ResponseQuality and ToolUsage) are not wired to online evaluation -- they are designed for **on-demand** use. On-demand evaluation lets you score a specific session by:

1. **Downloading span logs** from CloudWatch for a given session ID
2. **Calling the `evaluate()` API** on the data plane with those spans

This is useful for debugging specific conversations, testing evaluator rubrics, and investigating quality issues.

### Step 1: Invoke the agent and capture the session ID

We need a session ID from a recent invocation. Let's run one now and save it.

In [ ]:
import sys; sys.path.insert(0, '..')
from shared import test_agent

jwt_token = test_agent.get_test_token()

# Invoke with a prompt that exercises tools (good for ToolUsage evaluator)
result = test_agent.invoke(
    "Calculate the compound interest on $5,000 at 6% for 10 years, then create a task to review my investment portfolio",
    jwt_token=jwt_token,
)

eval_session_id = result["session_id"]
print(f"\nSession ID for evaluation: {eval_session_id}")

### Step 2: Download span logs from CloudWatch

The `evaluate()` API requires the raw span logs as input. We query CloudWatch Logs Insights to download all spans for the session.

> **Note:** It can take 1--2 minutes for spans to appear in CloudWatch after an invocation. If the query returns empty results, wait and re-run this cell.

In [ ]:
import boto3, json, time
from datetime import datetime, timedelta

region = utils.get_region()
runtime_config = utils.load_config("runtime")
runtime_id = runtime_config["runtime_id"]

logs_client = boto3.client("logs", region_name=region)

def query_logs(log_group_name, query_string):
    """Run a CloudWatch Logs Insights query and return results."""
    start_time = datetime.now() - timedelta(minutes=60)
    end_time = datetime.now()

    query_id = logs_client.start_query(
        logGroupName=log_group_name,
        startTime=int(start_time.timestamp()),
        endTime=int(end_time.timestamp()),
        queryString=query_string,
    )["queryId"]

    while True:
        result = logs_client.get_query_results(queryId=query_id)
        if result["status"] in ("Complete", "Failed"):
            break
        time.sleep(1)

    if result["status"] == "Failed":
        raise Exception("CloudWatch Logs Insights query failed")
    return result["results"]

def get_session_spans(session_id):
    """Download all span logs for a session from both log groups."""
    query = f"""fields @timestamp, @message
    | filter ispresent(scope.name) and ispresent(attributes.session.id)
    | filter attributes.session.id = "{session_id}"
    | sort @timestamp asc"""

    # Runtime log group
    runtime_log_group = f"/aws/bedrock-agentcore/runtimes/{runtime_id}-DEFAULT"
    runtime_results = query_logs(runtime_log_group, query)
    print(f"  Runtime spans: {len(runtime_results)}")

    # AWS vended spans log group
    aws_results = query_logs("aws/spans", query)
    print(f"  AWS spans:     {len(aws_results)}")

    # Extract JSON messages
    spans = []
    for row in runtime_results + aws_results:
        for field in row:
            if field["field"] == "@message" and field["value"].strip().startswith("{"):
                spans.append(json.loads(field["value"]))
    
    print(f"  Total spans:   {len(spans)}")
    return spans

print(f"Downloading spans for session: {eval_session_id[:16]}...")
print()
session_spans = get_session_spans(eval_session_id)

if not session_spans:
    print("\n⚠ No spans found yet. Wait 1-2 minutes and re-run this cell.")

### Step 3: Run the custom evaluators

Now we call the `evaluate()` **data plane** API with each custom evaluator. The API sends the span data to the LLM judge, which scores the session according to our rubric.

- **ResponseQuality** (SESSION level) — evaluates the entire conversation
- **ToolUsage** (TRACE level) — evaluates each individual turn where tools were used

In [ ]:
data_client = boto3.client("bedrock-agentcore", region_name=region)

# Look up custom evaluator IDs
eval_cfg = utils.load_config("evaluations")
custom_evaluators = eval_cfg.get("custom_evaluators", {})

# Get the actual IDs (they may have been saved as "existing" if created in a prior run)
# In that case, look them up from the control plane
all_evaluators = control.list_evaluators().get("evaluators", [])
evaluator_map = {e["evaluatorName"]: e["evaluatorId"] for e in all_evaluators}

rq_id = evaluator_map.get("ResponseQuality", custom_evaluators.get("ResponseQuality"))
tu_id = evaluator_map.get("ToolUsage", custom_evaluators.get("ToolUsage"))

print(f"ResponseQuality evaluator ID: {rq_id}")
print(f"ToolUsage evaluator ID:       {tu_id}")

def run_evaluation(evaluator_id, evaluator_name, spans):
    """Run on-demand evaluation and print results."""
    print(f"\n{'='*60}")
    print(f"Running {evaluator_name} evaluation...")
    print(f"{'='*60}")

    response = data_client.evaluate(
        evaluatorId=evaluator_id,
        evaluationInput={"sessionSpans": spans},
    )

    for result in response["evaluationResults"]:
        if result.get("errorCode"):
            print(f"\n  ERROR: {result['errorCode']} — {result.get('errorMessage', '')}")
            continue

        score = result.get("value", "N/A")
        label = result.get("label", "N/A")
        explanation = result.get("explanation", "")
        tokens = result.get("tokenUsage", {})
        context = result.get("context", {}).get("spanContext", {})

        print(f"\n  Score: {score} — {label}")
        if context.get("traceId"):
            print(f"  Trace: {context['traceId'][:32]}...")
        print(f"  Tokens: {tokens.get('inputTokens', 0)} in / {tokens.get('outputTokens', 0)} out")
        print(f"\n  Explanation:\n  {explanation[:500]}")

    return response["evaluationResults"]

# Run ResponseQuality (SESSION level)
rq_results = run_evaluation(rq_id, "ResponseQuality", session_spans)

# Run ToolUsage (TRACE level)
tu_results = run_evaluation(tu_id, "ToolUsage", session_spans)

### How on-demand evaluation works

The flow you just ran:

1. **Invoked the agent** — this generated OpenTelemetry spans that were written to CloudWatch Logs
2. **Downloaded spans** — queried CloudWatch Logs Insights for all spans matching the session ID, from both the Runtime log group and the `aws/spans` log group
3. **Called `evaluate()`** — passed the raw span data to the data plane API along with the evaluator ID. The service extracted the relevant context from the spans, filled in the evaluator's template placeholders, and sent everything to the judge model
4. **Received scores** — the LLM judge returned a numerical score, label, and explanation for each evaluation target (one result for SESSION-level, one per trace for TRACE-level)

On-demand evaluation is ideal for:
- **Testing evaluator rubrics** before deploying them as online evaluations
- **Investigating specific sessions** that received low online evaluation scores
- **Comparing evaluator versions** side by side on the same session data

## Evaluator design notes

### Custom evaluator levels and placeholders

Each evaluator level requires specific **placeholders** in the instructions. These placeholders are filled in automatically with data from agent traces:

| Level | Required placeholders (at least one) | Best for |
|---|---|---|
| **SESSION** | `{context}`, `{available_tools}`, `{actual_tool_trajectory}`, `{expected_tool_trajectory}`, `{assertions}` | Full conversation quality |
| **TRACE** | `{context}`, `{assistant_turn}`, `{expected_response}` | Individual turn quality |
| **TOOL_CALL** | *(built-in only)* | Tool selection accuracy |

Placeholders with `expected_` require **reference inputs** (ground truth) and can only be used for **on-demand** evaluation. For **online** evaluation (continuous monitoring), use built-in evaluators or custom evaluators that only reference actual data.

### Rating scale

Each evaluator defines a **numerical rating scale** with labels and definitions. The LLM judge outputs a score with justification. The `evaluatorConfig` uses the `llmAsAJudge` structure:

```python
evaluatorConfig={
    "llmAsAJudge": {
        "instructions": "...",  # Must include at least one placeholder
        "ratingScale": {
            "numerical": [
                {"value": 1, "label": "Poor", "definition": "..."},
                {"value": 5, "label": "Excellent", "definition": "..."},
            ]
        },
        "modelConfig": {
            "bedrockEvaluatorModelConfig": {
                "modelId": "us.anthropic.claude-sonnet-4-5-20250929-v1:0"
            }
        }
    }
}
```

### Online vs on-demand evaluation

| Mode | How it works | When to use |
|---|---|---|
| **Online** | Continuously samples live traffic from CloudWatch Logs | Production monitoring |
| **On-demand** | Evaluates specific traces/sessions by ID | Testing, investigation, debugging |

Online evaluations use the `create_online_evaluation_config` API (control plane) and require an IAM execution role with CloudWatch Logs read permissions and Bedrock model invocation permissions.

## What's next

All **9 AgentCore services** are now active for Aria:

1. **Runtime** -- Hosts the Strands agent
2. **Code Interpreter** -- Python execution sandbox
3. **Browser Tool** -- Web browsing capability
4. **Memory** -- Short-term and long-term memory (preferences, facts, summaries)
5. **Gateway** -- MCP protocol bridge to the Task API
6. **Identity** -- CUSTOM_JWT authentication via Cognito
7. **Policy** -- Cedar policy enforcement at the Gateway
8. **Observability** -- Automatic OTel tracing to CloudWatch
9. **Evaluations** -- LLM-as-judge quality monitoring

In **Module 8**, we review the full architecture, verify all services, and run integration tests across every capability.

In [ ]:
import sys; sys.path.insert(0, '..')
from shared.progress import show

show("07")

---

**Next up: [Module 8 -- Full Deployment Review](../08-full-deployment/notebook.ipynb)**